# building makemore（中文注释版）

这是 Andrej Karpathy《The spelled-out intro to language modeling: building makemore》第 1 讲的跟练笔记本。
目标：给一堆人名，训练一个**字符级 bigram 语言模型**（根据前 1 个字符预测下一个字符），再用它生成新名字。

本文件是在你原始 notebook 上**逐行加中文注释**得到的，**代码逻辑一行没改**。原始版本已备份为 `building makemore.BACKUP.ipynb`。
想看结构更完整、从 bigram 一路到 MLP、且已实跑验证的整理版，见同目录 `makemore_中文注释教学版.ipynb`。

学习主线：
1. 读数据 → 2. 字典统计 bigram → 3. 张量矩阵统计 → 4. 计数变概率 + 采样 → 5. 生成名字 →
6. 用 loss 评估 → 7. 改用神经网络(one-hot + softmax + 反向传播)学出同样的规律。

## 1. 读取数据

In [ ]:
# 下面两行是 IDE(PyCharm)自动补全时误加进来的,和本项目无关,还会报错,已注释掉(可直接删除)
# from networkx.algorithms.bipartite.basic import color
# from sympy.parsing.sympy_parser import null

words=open('names.txt','r').read().splitlines()  # 读入 names.txt 并按换行切分成列表,每个元素是一个名字

In [ ]:
words[:10]   # 看一眼前 10 个名字

In [ ]:
len(words)   # 一共有多少个名字(全量数据约 32000 个)

In [ ]:
min(len(w)for w in words)   # 最短名字的字符数

In [ ]:
max(len(w) for w in words)  # 最长名字的字符数

## 2. 用字典统计 bigram（最直观的版本）

**bigram** = 相邻的两个字符。先用一个普通 Python 字典 `b` 数一数每种字符对出现了多少次。
`<S>`/`<E>` 是开始/结束标记（后面会统一改用 `.`）。

In [ ]:
b={}                                   # 用字典统计每个 bigram(字符对)出现的次数
for w in words:                        # 遍历每个名字
    chs=['<S>']+list(w)+['<E>']        # 首尾加标记:<S>=开始, <E>=结束
    for ch1,ch2 in zip(chs, chs[1:]):  # 取相邻字符对:(<S>,e)(e,m)(m,m)(m,a)(a,<E>)
        bigram=(ch1,ch2)              # 组成一个 bigram 元组
        b[bigram]=b.get(bigram,0)+1   # 该 bigram 计数 +1(没出现过则从 0 开始)

In [ ]:
#出现的次数b
sorted(b.items(),key=lambda kv:-kv[1])   # 按出现次数从多到少排序,看最常见的字符组合

In [ ]:
b.items()   # 字典里所有的 (bigram, 次数) 键值对

In [ ]:
list(w)     # 把变量 w 拆成单字符列表(w 是上面循环遗留下来的最后一个名字)

In [ ]:
w           # 查看当前 w 的值(循环结束后残留的最后一个名字)

In [ ]:
w[1:]       # 从第 2 个字符开始的切片(演示切片,帮助理解 zip(chs, chs[1:]) 的错位配对)

## 3. 用张量矩阵统计 bigram

字典不方便做数学运算，改用一个 27×27 的**张量矩阵** `N`：`N[i, j]` = 字符 i 后面接字符 j 的次数。
27 = 26 个字母 + 1 个 `.`（统一用 `.` 表示开始和结束）。

In [ ]:
import  torch   # 导入 PyTorch(张量运算 + 自动求导)

In [ ]:
N=torch.zeros((27,27),dtype=torch.int32)   # 27x27 全零计数矩阵,N[i,j]=字符i后接字符j的次数

In [ ]:
chars=sorted(list(set(''.join(words))))    # 所有名字拼成一个大字符串,去重后排序 -> ['a','b',...,'z']
stoi={s:i+1 for i,s in enumerate(chars)}   # string->int:字母映射到 1~26(0 留给特殊符号)
stoi['.']=0                                 # '.' 作为开始/结束标记,编号 0
itos={i:s for s,i in stoi.items()}          # int->string:反向映射,采样时把编号翻译回字母

In [ ]:
for w in words:                        # 遍历每个名字
    chs=['.']+list(w)+['.']            # 首尾加 '.' 标记
    for ch1,ch2 in zip(chs, chs[1:]):  # 取相邻字符对
        ix1=stoi[ch1]                  # 前字符编号
        ix2=stoi[ch2]                  # 后字符编号
        N[ix1,ix2]+=1                  # 计数矩阵对应格子 +1

把计数矩阵画出来，颜色越深表示该字符组合越常见：

In [ ]:
import  matplotlib.pyplot as plt           # 画图库
%matplotlib inline                          # 让图像内嵌在 notebook 里显示
plt.figure(figsize=(16,16))                 # 设置一张大画布
plt.imshow(N,cmap='Blues')                  # 用蓝色深浅表示计数大小
for i in range(27):                         # 遍历每一行
    for j in range(27):                     # 遍历每一列
        chstr=itos[i]+itos[j]               # 该格子对应的字符对,如 'em'
        plt.text(j,i,chstr,ha='center',va='bottom',color='gray')       # 格子上方写字符对
        plt.text(j,i,N[i,j].item(),ha='center',va='top',color='gray')  # 格子下方写出现次数
plt.axis('off')                             # 隐藏坐标轴

## 4. 从计数到概率 + 采样原理

先拿第 0 行（即 `.` 开头之后）试验：把计数归一化成概率，就是**每个名字首字母的概率分布**。
然后用 `torch.multinomial` 按这个概率随机抽样。

In [ ]:
N[0,:]        # 第 0 行:'.'(开头)后面接各字符的原始计数
p=N[0].float()  # 转成浮点数
p=p/p.sum()     # 归一化成概率分布(每个字符作为首字母的概率)
p               # 查看这个概率分布

In [ ]:
g=torch.Generator().manual_seed(2147483647)                                 # 固定随机种子,保证可复现
ix=torch.multinomial(p,num_samples=1,replacement=True,generator=g).item()   # 按概率 p 抽 1 个字符编号
itos[ix]                                                                     # 翻译回字母,看抽到了啥

In [ ]:
g=torch.Generator().manual_seed(2147483647)   # 固定种子
p=torch.rand(3,generator=g)                    # 随便造一个长度为 3 的随机向量(演示用)
p=p/p.sum()                                     # 归一化成概率分布
p                                               # 查看

In [ ]:
torch.multinomial(p,num_samples=100,replacement=True,generator=g)   # 按概率 p 抽 100 次,验证抽样比例≈概率

In [ ]:
p.shape   # 查看形状

## 5. 归一化成概率矩阵 P

把整个 27×27 计数矩阵一次性归一化：**每一行**除以该行总和，得到 `P[i, j]` = 字符 i 之后是字符 j 的概率。

⚠️ 这里 `P=N.float()` **没有做 +1 平滑**。如果第 7 步算 loss 时出现 `inf`，就是因为某些 bigram 次数为 0、概率为 0、取 log 变成 -∞。
把下面 `P=(N+1).float()` 那行的注释去掉即可解决。

关键细节：`sum(1, keepdim=True)` 得到 (27,1)，除法时才会**按行广播**正确。

In [ ]:
P=N.float()                    # 计数矩阵转浮点(这里先不做平滑)
#模型平滑。P=(N+1).float()      # 平滑写法:每个计数 +1,避免出现概率为 0(取 log 变成 -inf)

#P=P/P.sum()
p                              # (这行是上面残留的变量 p,不是关键)

In [ ]:
P.shape   # 查看形状 (27, 27)

In [ ]:
P.sum(0,keepdim=True).shape #行   # 沿第 0 维求和(把每一列加起来),形状 (1,27)
P.sum(1,keepdim=True)#列          # 沿第 1 维求和(把每一行加起来),形状 (27,1) —— 归一化要用这个

In [ ]:
P=P/P.sum(1,keepdim=True) #广播定义   # 每行除以该行之和 -> 每行变成概率分布(keepdim 保证按行广播)

## 6. 用 bigram 概率生成名字

从 `.`（编号 0）开始，反复按当前字符行的概率分布采样下一个字符，抽到 `.` 就结束。

In [ ]:
g=torch.Generator().manual_seed(2147483647)   # 固定种子
for i in range(27):                            # 生成若干个名字
    out=[]                                      # 存放当前名字的字符
    ix=0                                        # 从 '.'(编号 0)开始
    while True:                                 # 循环采样直到遇到结束符
        #p=N[ix].float()                        # (注释掉的旧写法:每次现算概率)
        #p=p/p.sum()
        #p=torch.ones(27)                       # (若改用均匀分布会生成乱码,可对比体会模型作用)
        p=P[ix]                                 # 取'当前字符'这一行的概率分布
        ix=torch.multinomial(p,num_samples=1,replacement=True,generator=g).item()  # 按概率抽下一个字符
        out.append(itos[ix])                    # 记录字符
        if ix==0:                               # 抽到 '.'(结束符)
            break                               # 结束当前名字
    print(''.join(out))                         # 打印生成的名字

## 7. 评估：平均负对数似然 loss

衡量模型好坏：看它对**真实数据**给出的概率有多高。
把每个真实 bigram 的概率取 `log` 累加、取负、求平均，得到**平均负对数似然**（loss，越小越好，本模型约 2.45）。

> 若这里打印出 `inf`，回到第 5 步启用 `P=(N+1).float()` 平滑。

In [ ]:
log_likelihood=0.0                     # 累加对数似然
n=0                                     # 统计一共有多少个 bigram
for w in words:                         # 遍历每个名字
    chs=['.']+list(w)+['.']             # 首尾加 '.'
    for ch1,ch2 in zip(chs, chs[1:]):   # 遍历每个 bigram
        ix1=stoi[ch1]                   # 前字符编号
        ix2=stoi[ch2]                   # 后字符编号
        prob=P[ix1,ix2]                 # 模型给这个真实 bigram 的概率
        logprob=torch.log(prob)         # 取对数
        log_likelihood+=logprob         # 累加
        n+=1                            # 计数 +1
        #print(f'{ch1},{ch2}:{prob:.4f} {logprob:.4f}')   # (调试用:逐个打印)
print(f'{log_likelihood=}')             # 打印总对数似然
nll=-log_likelihood                     # 负对数似然(取负号,因为要最小化)
print(f'{nll=}')                        # 打印
print(f'{nll/n}')                       # 除以总数 = 平均 loss(评判标准)

## 8. 神经网络版：构造训练样本

同样是 bigram，但改用**神经网络 + 梯度下降**把规律学出来。这套流程(前向→loss→反向→更新)适用于所有深度学习模型。

⚠️ 注意 `words[:1]`：这里**只用了第 1 个名字**做演示，样本很少。要真正训练，把它改成 `words`(全量)即可。

In [ ]:
xs,ys=[],[]                            # xs=输入字符编号, ys=目标(下一个)字符编号
for w in words[:1]:                     # 只用第 1 个名字做演示(实战改成 words 全量)
    chs=['.']+list(w)+['.']             # 首尾加 '.'
    for ch1,ch2 in zip(chs, chs[1:]):   # 遍历 bigram
        ix1=stoi[ch1]                   # 输入字符编号
        ix2=stoi[ch2]                   # 目标字符编号
        xs.append(ix1)                  # 收集输入
        ys.append(ix2)                  # 收集目标
xs=torch.tensor(xs)                     # 转成张量
ys=torch.tensor(ys)                     # 转成张量

In [ ]:
xs.float()   # 查看输入(浮点显示)

In [ ]:
ys.float()   # 查看目标(浮点显示)

## 9. one-hot 编码 + 理解矩阵乘法

神经网络吃不了整数编号，要转成 **one-hot 向量**（只有对应位置是 1，其余是 0）。
`xenc @ w`（one-hot 乘权重矩阵）本质上就是**取出权重矩阵中对应的那一行**，得到每个字符的“得分”。

In [ ]:
import torch.nn.functional as F             # 常用函数模块(one_hot、cross_entropy 等)
xenc=F.one_hot(xs,num_classes=27).float()   # 把输入编号变成 one-hot 向量 (样本数, 27)

In [ ]:
xenc.shape   # (5, 27):5 个样本,每个是 27 维 one-hot

In [ ]:
plt.imshow(xenc,cmap='Blues')   # 可视化 one-hot:每行只有一个格子是 1

In [ ]:
xenc.dtype   # 数据类型(float32)

In [ ]:
w=torch.randn((27,27))   # 随机权重矩阵 27x27(演示用,注意这里是小写 w)
xenc @ w                  # one-hot 乘权重 = 取出对应行,得到每个字符的'得分'(logits)

In [ ]:
(xenc @ w).shape   # 查看形状 (5, 27)

In [ ]:
(xenc @ w)[3,13]   # 第 4 个样本对第 14 个字符的得分

In [ ]:
xenc[3]            # 第 4 个样本的 one-hot 向量

In [ ]:
w[:,13]            # 权重矩阵的第 14 列(验证上面的点积)

## 10. softmax：把“得分”变成“概率”

得分(logits)可正可负。先 `exp()` 变成正数(相当于'计数')，再按行归一化，就得到概率分布。
这两步合起来就是 **softmax**。

In [ ]:
(xenc @ w).exp()   # 得分指数化 -> 全为正数,相当于'计数'

In [ ]:
logits=xenc@w                             # 得分(log-counts)
counts=logits.exp()                       # 指数化成正数(counts)
probs=counts/counts.sum(1,keepdim=True)   # 按行归一化成概率(exp + 归一化 = softmax)
probs                                     # 查看概率

In [ ]:
probs[0].sum()   # 每行概率之和应约等于 1

## 11. 手动拆解 loss，理解负对数似然

逐个样本看：模型给“正确目标字符”的概率是多少，取负对数就是它的损失。

In [ ]:
nlls=torch.zeros(5)              # 存 5 个样本各自的负对数似然
for i in range(5):               # 遍历 5 个样本
    x=xs[i].item()               # 输入字符编号
    y=ys[i].item()               # 目标字符编号
    print(f'{itos[x],{itos[y]}}')  # 打印 (输入字符, 目标字符)
    print("output",probs[i])     # 模型输出的整行概率分布
    p=probs[i,y]                 # 模型给'正确目标 y'的概率
    print(p.item())              # 打印该概率
    logp=torch.log(p)            # 取对数
    print(logp.item())           # 打印
    nll=-logp                    # 负对数似然
    print('negative',nll.item()) # 打印
    nlls[i]=nll                  # 记录
    print(nlls.mean().item())    # 打印到目前为止的平均 loss

## 12. 前向传播（开启梯度追踪）

把权重换成大写 `W` 并设 `requires_grad=True`，PyTorch 就会追踪它、为它计算梯度。

In [ ]:
g=torch.Generator().manual_seed(2147483647)             # 固定种子
W=torch.randn((27,27),generator=g,requires_grad=True)   # 权重矩阵(大写W),requires_grad=True 表示需要梯度
xenc=F.one_hot(xs,num_classes=27).float()               # 输入 one-hot
logits=xenc@W                                           # 前向:得分
counts=logits.exp()                                     # 指数化
probs=counts/counts.sum(1,keepdim=True)                 # softmax 概率
probs                                                   # 查看

In [ ]:
probs.shape   # 查看形状 (5, 27)

In [ ]:
#probs[0,5],probs[1,13],probs[2,13],probs[3,1],probs[4,0]   # (手动取每个样本正确目标的概率,已注释)

In [ ]:
loss=-probs[torch.arange(5),ys].log().mean()   # 取 5 个样本正确目标的概率 -> log -> 取负 -> 求平均 = loss
loss                                            # 查看 loss(它带着计算图,可反向传播)

In [ ]:
print(loss.item())   # 打印 loss 数值

## 13. 反向传播 + 更新参数

`loss.backward()` 自动算出 `W` 每个元素的梯度；再沿梯度**反方向**更新，就能让 loss 下降。

In [ ]:
W.grad   # 反向传播之前,梯度是 None

In [ ]:
#backward pass
W.grad=None       # 先清空梯度(PyTorch 梯度默认累加,每轮要清零)
loss.backward()   # 反向传播:自动计算 W 的梯度

In [ ]:
W.data+=-0.1*W.grad   # 沿梯度反方向更新一次(学习率 0.1)

## 14. 完整训练循环

把前向、算 loss、反向、更新串成循环，反复迭代让 loss 持续下降。`+0.01*(W**2).mean()` 是 L2 正则，作用相当于平滑。

In [ ]:
xs,ys=[],[]                            # 重新构造训练样本
for w in words[:1]:                     # 仍只用第 1 个名字做演示(实战改成 words 全量)
    chs=['.']+list(w)+['.']             # 首尾加 '.'
    for ch1,ch2 in zip(chs, chs[1:]):   # 遍历 bigram
        ix1=stoi[ch1]                   # 输入编号
        ix2=stoi[ch2]                   # 目标编号
        xs.append(ix1)                  # 收集输入
        ys.append(ix2)                  # 收集目标
xs=torch.tensor(xs)                     # 转张量
ys=torch.tensor(ys)                     # 转张量
num=xs.nelement()                       # 训练样本总数
print(num)                              # 打印样本数

g=torch.Generator().manual_seed(2147483647)             # 固定种子
W=torch.randn((27,27),generator=g,requires_grad=True)   # 初始化权重

In [ ]:
for i in range(100):                                       # 训练 100 轮
    xenc=F.one_hot(xs,num_classes=27).float()              # 前向:输入 one-hot
    logits=xenc@W                                          # 得分
    counts=logits.exp()                                    # 指数化
    probs=counts/counts.sum(1,keepdim=True)                # softmax 概率
    loss=-probs[torch.arange(num),ys].log().mean()+0.01*(W**2).mean()  # 平均负对数似然 + L2 正则(平滑)
    print(loss.item())                                     # 打印当前 loss(应逐轮下降)
    W.grad=None                                            # 清空梯度
    loss.backward()                                        # 反向传播
    W.data+=-50*W.grad                                     # 梯度下降更新(学习率 50)

## 15. 下一步

上面的训练只用了 `words[:1]`（第 1 个名字）做演示。要得到真正能用的模型：

1. 把第 8 步和第 14 步里的 `words[:1]` 改成 `words`（全量约 3.2 万个名字）。
2. 训练循环轮数适当加大（如 200+）。
3. 训练完后，用学到的 `W` 前向计算概率来采样生成名字（写法见第 6 步，把 `P[ix]` 换成对 one-hot 做 `xenc@W` 再 softmax）。

完整、可直接运行、并延伸到 MLP 的整理版见同目录 **`makemore_中文注释教学版.ipynb`**。
